## 1) Run training (including model initialisation) multiple times
Are the learning curves and final performance the same?

## 2) Evaluate the trained model on data from other participants
How well does it generalise?

## 3) Experiment with different hyperparameters
- Learning rates (lr parameter in optimiser): try e.g. 1e1, 1e-1, 1e-3, 1e-5
- Model parameters: See the braindecode documentation for parameter descriptions.

## 4) Try a different model
- Check the braindecode documentation and pick one.
- Try the provided fully connected EEGMLP model. How many neurons and layers can you use to get a similar parameter count as we had with the convolutional ShallowFBCSPNet (~7k)? How good is the performance?

In [ ]:
from torch import nn


class EEGMLP(nn.Module):
    def __init__(
        self,
        n_chans,
        n_outputs,
        n_times,
        n_layers=3,
        units_per_layer=256,
    ):
        super().__init__()

        if n_layers < 0:
            raise ValueError("n_layers must be non-negative.")

        if isinstance(units_per_layer, int):
            widths = [units_per_layer] * n_layers
        else:
            widths = list(units_per_layer)
            if len(widths) != n_layers:
                raise ValueError("units_per_layer must have exactly n_layers entries.")

        if any(width <= 0 for width in widths):
            raise ValueError("Hidden layer widths must be positive.")

        self.n_chans = n_chans
        self.n_times = n_times
        self.n_outputs = n_outputs

        # Flatten input
        layers = [nn.Flatten(start_dim=1)]

        # Add hidden layers
        in_features = n_chans * n_times
        for width in widths:
            layers.extend(
                [
                    nn.Linear(in_features, width),
                    nn.ReLU(),
                ]
            )
            in_features = width

        # Add the final output layer (prediction head)
        layers.append(nn.Linear(in_features, n_outputs))

        # Combine all layers into a sequential module
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # Also accept (batch, channels, time, 1).
        if x.ndim == 4 and x.shape[-1] == 1:
            x = x.squeeze(-1)

        if x.ndim != 3 or x.shape[1:] != (self.n_chans, self.n_times):
            raise ValueError(
                f"Expected (batch, {self.n_chans}, {self.n_times}), "
                f"got {tuple(x.shape)}."
            )

        return self.network(x)

In [ ]:
model = EEGMLP(
    n_chans=22,
    n_times=400,
    n_outputs=4,
    n_layers=2,
    units_per_layer=[16, 2],  # Number of units in each layer
)

print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5) Try different data preprocessing
